In [2]:
import cv2
import numpy as np
import os
import random
from tqdm import tqdm

def apply_motion_blur(image: np.ndarray, size: int, angle: int) -> np.ndarray:
    kernel = np.zeros((size, size), dtype=np.float32)
    kernel[(size - 1) // 2, :] = np.ones(size, dtype=np.float32)
    center = (size // 2, size // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    kernel = cv2.warpAffine(kernel, rotation_matrix, (size, size))
    kernel /= np.sum(kernel)
    return cv2.filter2D(image, -1, kernel)

def apply_defocus_blur(image: np.ndarray, radius: int) -> np.ndarray:
    kernel = np.zeros((radius * 2, radius * 2), dtype=np.float32)
    cv2.circle(kernel, (radius, radius), radius, (1, 1, 1), -1, cv2.LINE_AA)
    kernel /= np.sum(kernel)
    return cv2.filter2D(image, -1, kernel)

def apply_gaussian_blur(image: np.ndarray, size: int) -> np.ndarray:
    return cv2.GaussianBlur(image, (size, size), 0)

def main():
    # Enter location of clean and blur data
    input_dir = "./../Dataset/ProcessedDataset/CleanData"
    output_dir = "./../Dataset/ProcessedDataset/BlurredData"
    num_versions = 1

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created output directory: {output_dir}")

    sharp_images = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not sharp_images:
        print(f"No images found in {input_dir}. Please check the path.")
        return

    print(f"Found {len(sharp_images)} sharp images. Generating {num_versions} blurred version(s) for each.")

    for filename in tqdm(sharp_images, desc="Processing images"):
        try:
            sharp_path = os.path.join(input_dir, filename)
            image = cv2.imread(sharp_path)

            if image is None:
                print(f"Warning: Could not read {filename}. Skipping.")
                continue

            for i in range(num_versions):
                blur_type = random.choice(['motion', 'defocus', 'gaussian'])

                if blur_type == 'motion':
                    size = random.choice(range(5, 30, 2)) 
                    angle = random.randint(0, 180)
                    blurred_image = apply_motion_blur(image, size, angle)

                elif blur_type == 'defocus':
                    radius = random.randint(3, 15)
                    blurred_image = apply_defocus_blur(image, radius)

                else:  # gaussian
                    size = random.choice(range(5, 30, 2))
                    blurred_image = apply_gaussian_blur(image, size)
                    
                output_filename = filename
                output_path = os.path.join(output_dir, output_filename)
                cv2.imwrite(output_path, blurred_image)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    print("\nSynthetic blur generation complete.")
    print(f"Blurred images saved to: {output_dir}")

if __name__ == "__main__":
    main()

Found 300 sharp images. Generating 1 blurred version(s) for each.


Processing images:   0%|          | 0/300 [00:00<?, ?it/s]

Processing images: 100%|██████████| 300/300 [00:06<00:00, 47.35it/s]


Synthetic blur generation complete.
Blurred images saved to: ./../Dataset/ProcessedDataset/BlurredData
